In [1]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import TensorBoard

2025-06-13 15:49:51.326966: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749809991.339080   10872 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749809991.342709   10872 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1749809991.352540   10872 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1749809991.352553   10872 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1749809991.352555   10872 computation_placer.cc:177] computation placer alr

In [2]:
dataset = "../../asserts/datasets/data_1.csv"
model_save_path = "../../asserts/models/model_0_2.keras"
log_dir = "../../logs/"
lables= [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,15,26]

In [3]:
tb_callback = TensorBoard(log_dir)
# Model checkpoint callback
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    model_save_path, verbose=1, save_weights_only=False)
# Callback for early stopping
es_callback = tf.keras.callbacks.EarlyStopping(patience=20, verbose=1)

In [4]:
x_data = np.loadtxt(dataset,delimiter=",",dtype='float32',usecols=list(range(1,43)))

In [5]:
y_data = np.loadtxt(dataset,delimiter=",",dtype='int32',usecols=list(range(1)))

In [6]:
y_data = tf.keras.utils.to_categorical(y_data)

In [7]:
x_train,x_test,y_train,y_test = train_test_split(x_data,y_data,train_size=0.8,random_state=42)

In [8]:
y_test.shape

(4331, 27)

In [18]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Input(x_data[0].shape),
    tf.keras.layers.Dense(120,activation='relu'),
    tf.keras.layers.Dense(120,activation='relu'),
    tf.keras.layers.Dense(60,activation='relu'),
    tf.keras.layers.Dense(40,activation='relu'),
    tf.keras.layers.Dense(y_data.shape[1],activation='softmax')
])

In [19]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 120)            │         5,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 120)            │        14,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 60)             │         7,260 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 40)             │         2,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 27)             │         1,107 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,487 (119.09 KB)

 Trainable params: 30,487 (119.09 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
model.compile(optimizer="Adam",loss="categorical_crossentropy",metrics=["categorical_accuracy"])

In [21]:
%%time
model.fit(
    x_train,
    y_train,
    epochs=1000,
    batch_size=128,
    validation_data=(x_test, y_test),
    callbacks=[es_callback,cp_callback,tb_callback]
)

Epoch 1/1000
136/136 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - categorical_accuracy: 0.2807 - loss: 2.6145
Epoch 1: saving model to ../../asserts/models/model_0_2.keras
136/136 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step - categorical_accuracy: 0.2823 - loss: 2.6090 - val_categorical_accuracy: 0.8227 - val_loss: 0.6994
Epoch 2/1000
130/136 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - categorical_accuracy: 0.8723 - loss: 0.5707
Epoch 2: saving model to ../../asserts/models/model_0_2.keras
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - categorical_accuracy: 0.8734 - loss: 0.5668 - val_categorical_accuracy: 0.9192 - val_loss: 0.3741
Epoch 3/1000
128/136 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - categorical_accuracy: 0.9254 - loss: 0.3718
Epoch 3: saving model to ../../asserts/models/model_0_2.keras
136/136 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - categorical_accuracy: 0.9258 - loss: 0.3693 - val_categorical_accuracy: 0.9342 - val_loss: 0.2936
Epoch 4/1000
124/136 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - categorical_accuracy: 0.9392 - l

In [22]:
val_loss, val_acc = model.evaluate(x_test, y_test, batch_size=128)

34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - categorical_accuracy: 0.9883 - loss: 0.0526


In [23]:
model = tf.keras.models.load_model(model_save_path)


In [24]:
t =model.predict(x_test)

136/136 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step


In [25]:
x= {0:"A",1:"B",2:"C",3:"D",4:"E",5:"F",6:"G",7:"H",8:"I",9:"J",10:"K",11:"L",12:"M",13:"N",14:"O",15:"P",16:"Q",17:"R",18:"S",19:"T",20:"U",21:"V",22:"W",23:"X",24:"Y",25:"Z",26:"space"}

In [26]:
count = 0
for z,_ in enumerate(t):
    if ([np.argmax(t[z])]!=[np.argmax(y_test[z])]):
        print(f"{x[np.argmax(t[z])]},{x[np.argmax(y_test[z])]}")

        count+=1
count  

K,J
N,O
M,N
H,I
K,L
Z,space
N,O
H,I
M,N
W,V
H,I
G,H
W,V
O,P
F,G
A,Y
W,V
U,V
E,F
X,Y
U,V
H,I
D,E
H,I
G,H
R,S
W,X
K,L
C,D
W,V
P,Q
D,E
F,G
D,C
Q,R
W,V
E,space
C,D
I,J
B,C
K,L
M,N
J,K
W,V
Z,space
K,L
Y,Z
U,T
V,W
L,T
X,K
D,E
E,F
N,M


54

In [47]:
y_test.shape

(2166, 27)

In [20]:
!uv add seaborn
!uv add pandas

Resolved 165 packages in 1ms
Audited 160 packages in 0.07ms
Resolved 165 packages in 1ms
Audited 160 packages in 0.06ms


In [23]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

def print_confusion_matrix(y_true, y_pred, report=True):
    labels = np.sort(np.unique(y_true))
    cmx_data = confusion_matrix(y_true, y_pred, labels=labels)
    
    df_cmx = pd.DataFrame(cmx_data, index=labels, columns=labels)
 
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(df_cmx, annot=True, fmt='g' ,square=False)
    ax.set_ylim(len(set(list(y_true))), 0)
    plt.show()
    
    if report:
        print('Classification Report')
        print(classification_report(y_test, y_pred))

Y_pred = model.predict(x_test)
y_pred = np.argmax(Y_pred, axis=1)

print_confusion_matrix(y_test, y_pred)

68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


ValueError: Classification metrics can't handle a mix of multilabel-indicator and multiclass targets

In [24]:
model.save(model_save_path)